# LSTM

## Import libraries

In [90]:
import ccxt
from datetime import datetime
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_classif

## Constants

In [71]:
CLIENT = ccxt.binance()
TICKER = "BTC/USDT"
END_DATE = datetime.now().timestamp() * 1000
START_DATE = CLIENT.parse8601("2010-11-03T20:00:00Z")
TIMEFRAME = "1d"
LAGS_NUM = 240
TRAIN_TEST_SPLIT_RATIO = 0.8
HIDDEN_NEURONS_NUM = 25
EPOCHS_NUM = 100
BATCH_SIZE = 32
DROPOUT_RATE = 0.1

In [72]:
TIMEFRAME_DICT = {
    "1m": 60 * 1000,
    "5m": 5 * 60 * 1000,
    "15m": 15 * 60 * 1000,
    "1h": 60 * 60 * 1000,
    "1d": 24 * 60 * 60 * 1000,
}

FREQ_DICT = {
    "1m": "T",
    "5m": "5T",
    "15m": "15T",
    "1h": "H",
    "1d": "D",
}

## Fetching data

In [73]:
def fetch_data(client, ticker, start_date, end_date, timeframe):
    data = []
    while start_date < end_date:
        data = data + client.fetch_ohlcv(
            ticker,
            timeframe=timeframe,
            since=start_date,
            limit=1000,
        )
        start_date = data[-1][0] + TIMEFRAME_DICT[timeframe]
    return data

def preprocess_data(data):
    data = pd.DataFrame(
        data,
        columns=["Date", "Open", "High", "Low", "Close", "Volume"]
    )
    data["Date"] = pd.to_datetime(data["Date"], unit="ms")
    data = data.set_index("Date")
    return data

In [74]:
data = fetch_data(CLIENT, TICKER, START_DATE, END_DATE, TIMEFRAME)
data = preprocess_data(data)
data.head()

,Open,High,Low,Close,Volume
Date,,,,,
2017-08-17,4261.48,4485.39,4200.74,4285.08,795.150377
2017-08-18,4285.08,4371.52,3938.77,4108.37,1199.888264
2017-08-19,4108.37,4184.69,3850.00,4139.98,381.309763
2017-08-20,4120.98,4211.08,4032.62,4086.29,467.083022
2017-08-21,4069.13,4119.62,3911.79,4016.00,691.743060


### Verifying duplicates

In [75]:
data.index.duplicated().sum()

0

### Verifying continuous of data

In [76]:
all_dates = pd.date_range(start=data.index.min(), end=data.index.max(), freq=FREQ_DICT[TIMEFRAME])
missing = all_dates.difference(data.index)
print(len(missing), missing)   # 0 => none missing dates

0 DatetimeIndex([], dtype='datetime64[ms]', freq='D')


## Processing data

In [77]:
data['log_return_close'] = np.log(data['Close'].shift(1) / data['Close'].shift(2))
data['log_return_open'] = np.log(data['Open'].shift(1) / data['Open'].shift(2))
data['log_return_high'] = np.log(data['High'].shift(1) / data['High'].shift(2))
data['log_return_low'] = np.log(data['Low'].shift(1) / data['Low'].shift(2))

In [78]:
data['log_spread_hl'] = np.log(data['High'].shift(1) / data['Low'].shift(1))
data['log_spread_co'] = np.log(data['Close'].shift(1) / data['Open'].shift(1))

In [79]:
data['moving_average_return_close'] = data['log_return_close'].rolling(window=LAGS_NUM).mean()
data['deviation_return_close'] = data['log_return_close'].rolling(window=LAGS_NUM).std()

In [80]:
data['volatility_ratio'] = data['log_return_close'].rolling(window=LAGS_NUM).std() / data['log_return_close'].rolling(window=LAGS_NUM).mean()

In [81]:
data['true_range'] = np.maximum(data['High'].shift(1) - data['Low'].shift(1), np.abs(data['High'].shift(1) - data['Close'].shift(2)), np.abs(data['Low'].shift(1) - data['Close'].shift(2)))
data['average_true_range'] = data['true_range'].rolling(window=LAGS_NUM).mean()

In [82]:
data['velocity'] = data['Close'].shift(1) - data['Close'].shift(2)
data['acceleration'] = data['velocity'] - data['velocity'].shift(1)

In [83]:
data['volume_mean'] = data['Volume'].shift(1).rolling(window=LAGS_NUM).mean()
data['volume_std'] = data['Volume'].shift(1).rolling(window=LAGS_NUM).std()
data['delta_volume'] = (data['Volume'].shift(1) - data['volume_mean']) / data['volume_std']

In [84]:
data['return_close'] = (data['Close'] - data['Close'].shift(1)) / data['Close'].shift(1)

In [85]:
data['return_close_binary'] = (data['return_close'] > 0).astype(int)

In [86]:
data = data.dropna()
data.head()

,Open,High,Low,Close,Volume,log_return_close,log_return_open,log_return_high,log_return_low,log_spread_hl,...,volatility_ratio,true_range,average_true_range,velocity,acceleration,volume_mean,volume_std,delta_volume,return_close,return_close_binary
Date,,,,,,,,,,,,,,,,,,,,,
2018-04-15,8004.00,8429.54,7999.02,8355.00,27946.720444,0.015319,-0.005761,-0.005772,0.010037,0.047020,...,22.926130,376.00,937.504625,121.60,167.19,18332.126003,19853.215278,0.669371,0.044504,1
2018-04-16,8355.07,8419.00,7867.00,8064.92,36664.069715,0.043542,0.015933,0.029317,0.023914,0.052423,...,20.155451,430.53,937.495375,355.99,234.39,18443.571137,19831.701255,0.479190,-0.034719,0
2018-04-17,8064.92,8173.70,7825.40,7885.02,32152.603567,-0.035336,0.042927,-0.001251,-0.016642,0.067814,...,21.473320,552.00,938.400833,-290.08,-646.07,18594.749304,19831.728273,0.911132,-0.022306,0
2018-04-18,7890.96,8236.43,7868.00,8173.00,26969.044570,-0.022559,-0.035345,-0.029569,-0.005302,0.043547,...,21.788849,348.30,939.108500,-179.90,110.18,18726.772306,19816.006122,0.677525,0.036522,1
2018-04-19,8173.99,8296.00,8080.00,8278.00,27113.847464,0.035871,-0.021806,0.007645,0.005429,0.045763,...,20.164874,368.43,939.777667,287.98,467.88,18836.261062,19788.516309,0.410985,0.012847,1


In [87]:
X = data.drop(columns=['return_close_binary', 'return_close', 'High', 'Low', 'Open', 'Close', 'Volume'])
y = data['return_close_binary']

In [88]:
X.head()

,log_return_close,log_return_open,log_return_high,log_return_low,log_spread_hl,log_spread_co,moving_average_return_close,deviation_return_close,volatility_ratio,true_range,average_true_range,velocity,acceleration,volume_mean,volume_std,delta_volume
Date,,,,,,,,,,,,,,,,
2018-04-15,0.015319,-0.005761,-0.005772,0.010037,0.047020,0.015310,0.002601,0.059625,22.926130,376.00,937.504625,121.60,167.19,18332.126003,19853.215278,0.669371
2018-04-16,0.043542,0.015933,0.029317,0.023914,0.052423,0.042919,0.002958,0.059613,20.155451,430.53,937.495375,355.99,234.39,18443.571137,19831.701255,0.479190
2018-04-17,-0.035336,0.042927,-0.001251,-0.016642,0.067814,-0.035345,0.002778,0.059663,21.473320,552.00,938.400833,-290.08,-646.07,18594.749304,19831.728273,0.911132
2018-04-18,-0.022559,-0.035345,-0.029569,-0.005302,0.043547,-0.022559,0.002739,0.059677,21.788849,348.30,939.108500,-179.90,110.18,18726.772306,19816.006122,0.677525
2018-04-19,0.035871,-0.021806,0.007645,0.005429,0.045763,0.035118,0.002961,0.059701,20.164874,368.43,939.777667,287.98,467.88,18836.261062,19788.516309,0.410985


In [100]:
X_new = SelectKBest(f_classif, k=8).fit(X, y).get_feature_names_out(X.columns)


In [103]:
X_new

array(['log_return_close', 'log_return_low', 'log_spread_hl',
       'log_spread_co', 'average_true_range', 'velocity', 'acceleration',
       'volume_std'], dtype=object)